# PERSUADE v7: Span Ablation Analysis

**Goal:** Measure *where* in the prior context specific word choices depend on.

**Complementary to truncation:**
- Truncation (existing): How far back does useful context extend?
- Span ablation (new): Which part of prior context drives specific word choices?

**Method:**
- For each essay, compute baseline NLL on scoring region with full context
- Then ablate (delete) contiguous spans and measure ΔNLL
- Higher ΔNLL = that span was more important for predictions

**Spans tested (10% of context each):**
- Early: 0-10%
- Early-mid: 10-20%
- Middle: 40-50%
- Late: 70-80%
- Random: one random 10% span (seeded)

**Ablation method:** Option A (deletion) - remove tokens and concatenate remaining context.

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import math
import os
import time
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'score_long_span_ablation'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Scoring parameters (same as truncation experiment)
BURN_IN = 512  # EXTENDED regime
MAX_SCORE_TOKENS = 256

# Span ablation parameters
SPAN_FRACTION = 0.10  # 10% of context
MIN_SPAN_TOKENS = 32   # Minimum span size
MAX_SPAN_TOKENS = 128  # Maximum span size

# Span locations (as fraction of context)
SPAN_CONFIGS = {
    'early': (0.00, 0.10),
    'early_mid': (0.10, 0.20),
    'middle': (0.40, 0.50),
    'late': (0.70, 0.80),
    # 'random' handled separately
}

RANDOM_SEED = 42

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nSpan ablation config:")
print(f"  Span size: {SPAN_FRACTION*100:.0f}% of context")
print(f"  Min/max tokens: [{MIN_SPAN_TOKENS}, {MAX_SPAN_TOKENS}]")
print(f"  Spans: {list(SPAN_CONFIGS.keys())} + random")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization (T4 mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16 (A100 mode)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

## 2. Core Functions

In [ ]:
@torch.no_grad()
def compute_token_nlls(token_ids, target_start, target_end):
    """
    Compute NLL for each token in [target_start, target_end).
    
    Returns:
        list of (position, nll) tuples
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return []
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    nlls = []
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        nll = -log_probs[token_ids[i + 1]].item()
        nlls.append((i, nll))
    
    return nlls


@torch.no_grad()
def compute_top1_predictions(token_ids, target_start, target_end):
    """
    Get top-1 predicted token at each position in [target_start, target_end).
    
    Returns:
        list of (position, top1_token_id) tuples
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return []
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    predictions = []
    for i in range(target_start, target_end - 1):
        top1 = logits[i].argmax().item()
        predictions.append((i, top1))
    
    return predictions


def compute_span_boundaries(context_length, span_frac_start, span_frac_end):
    """
    Compute actual token indices for a span.
    
    Returns:
        (start_idx, end_idx) in token space
    """
    span_start = int(context_length * span_frac_start)
    span_end = int(context_length * span_frac_end)
    
    # Enforce min/max span size
    span_size = span_end - span_start
    if span_size < MIN_SPAN_TOKENS:
        span_end = min(span_start + MIN_SPAN_TOKENS, context_length)
    elif span_size > MAX_SPAN_TOKENS:
        span_end = span_start + MAX_SPAN_TOKENS
    
    return span_start, span_end


def ablate_span_deletion(token_ids, span_start, span_end):
    """
    Option A: Delete span and concatenate remaining tokens.
    
    Returns:
        ablated token_ids list
    """
    return token_ids[:span_start] + token_ids[span_end:]


print("Core functions defined")

In [ ]:
def run_span_ablation_for_essay(token_ids, essay_id, rng):
    """
    Run span ablation analysis for a single essay.
    
    Args:
        token_ids: list of token IDs
        essay_id: for tracking
        rng: random.Random instance for reproducibility
    
    Returns:
        list of result dicts (one per span condition)
    """
    n_tokens = len(token_ids)
    
    # Context is [0, BURN_IN), scoring region is [BURN_IN, BURN_IN + MAX_SCORE_TOKENS)
    context_length = BURN_IN
    target_start = BURN_IN
    target_end = min(n_tokens, BURN_IN + MAX_SCORE_TOKENS)
    
    if n_tokens < target_start + 10:
        return []  # Not enough tokens
    
    # ============================================================
    # Baseline: full context
    # ============================================================
    baseline_tokens = token_ids[:target_end]
    baseline_nlls = compute_token_nlls(baseline_tokens, target_start, target_end)
    baseline_top1 = compute_top1_predictions(baseline_tokens, target_start, target_end)
    
    if not baseline_nlls:
        return []
    
    baseline_mean_nll = np.mean([nll for _, nll in baseline_nlls])
    baseline_top1_dict = {pos: tok for pos, tok in baseline_top1}
    true_tokens = {pos: token_ids[pos + 1] for pos, _ in baseline_nlls}
    
    results = []
    
    # ============================================================
    # Fixed spans
    # ============================================================
    for span_label, (frac_start, frac_end) in SPAN_CONFIGS.items():
        span_start, span_end = compute_span_boundaries(context_length, frac_start, frac_end)
        
        # Ablate
        ablated_context = ablate_span_deletion(token_ids[:context_length], span_start, span_end)
        
        # Reconstruct full sequence for scoring
        # After ablation, context is shorter. We need to append target tokens.
        # The target tokens are the same (token_ids[target_start:target_end])
        ablated_full = ablated_context + token_ids[target_start:target_end]
        
        # New target positions (shifted due to deletion)
        new_target_start = len(ablated_context)
        new_target_end = len(ablated_full)
        
        # Compute NLLs on ablated sequence
        ablated_nlls = compute_token_nlls(ablated_full, new_target_start, new_target_end)
        ablated_top1 = compute_top1_predictions(ablated_full, new_target_start, new_target_end)
        
        if not ablated_nlls:
            continue
        
        ablated_mean_nll = np.mean([nll for _, nll in ablated_nlls])
        delta_nll = ablated_mean_nll - baseline_mean_nll
        
        # Compute flip rate
        n_flips = 0
        n_total = 0
        ablated_top1_list = [tok for _, tok in ablated_top1]
        baseline_top1_list = [baseline_top1_dict.get(pos) for pos, _ in baseline_nlls]
        
        for i, (b_tok, a_tok) in enumerate(zip(baseline_top1_list, ablated_top1_list)):
            if b_tok is not None and a_tok is not None:
                n_total += 1
                if b_tok != a_tok:
                    n_flips += 1
        
        flip_rate = n_flips / n_total if n_total > 0 else np.nan
        
        results.append({
            'essay_id': essay_id,
            'span_label': span_label,
            'span_start_token': span_start,
            'span_end_token': span_end,
            'span_size': span_end - span_start,
            'context_length': context_length,
            'baseline_mean_nll': baseline_mean_nll,
            'ablated_mean_nll': ablated_mean_nll,
            'delta_nll_mean': delta_nll,
            'flip_rate': flip_rate,
            'n_scored_tokens': len(ablated_nlls),
        })
    
    # ============================================================
    # Random span
    # ============================================================
    span_size = int(context_length * SPAN_FRACTION)
    span_size = max(MIN_SPAN_TOKENS, min(MAX_SPAN_TOKENS, span_size))
    
    # Random start position
    max_start = context_length - span_size
    if max_start > 0:
        span_start = rng.randint(0, max_start)
        span_end = span_start + span_size
        
        # Ablate
        ablated_context = ablate_span_deletion(token_ids[:context_length], span_start, span_end)
        ablated_full = ablated_context + token_ids[target_start:target_end]
        
        new_target_start = len(ablated_context)
        new_target_end = len(ablated_full)
        
        ablated_nlls = compute_token_nlls(ablated_full, new_target_start, new_target_end)
        ablated_top1 = compute_top1_predictions(ablated_full, new_target_start, new_target_end)
        
        if ablated_nlls:
            ablated_mean_nll = np.mean([nll for _, nll in ablated_nlls])
            delta_nll = ablated_mean_nll - baseline_mean_nll
            
            # Flip rate
            n_flips = 0
            n_total = 0
            ablated_top1_list = [tok for _, tok in ablated_top1]
            baseline_top1_list = [baseline_top1_dict.get(pos) for pos, _ in baseline_nlls]
            
            for b_tok, a_tok in zip(baseline_top1_list, ablated_top1_list):
                if b_tok is not None and a_tok is not None:
                    n_total += 1
                    if b_tok != a_tok:
                        n_flips += 1
            
            flip_rate = n_flips / n_total if n_total > 0 else np.nan
            
            results.append({
                'essay_id': essay_id,
                'span_label': 'random',
                'span_start_token': span_start,
                'span_end_token': span_end,
                'span_size': span_size,
                'context_length': context_length,
                'baseline_mean_nll': baseline_mean_nll,
                'ablated_mean_nll': ablated_mean_nll,
                'delta_nll_mean': delta_nll,
                'flip_rate': flip_rate,
                'n_scored_tokens': len(ablated_nlls),
            })
    
    return results

print("Span ablation function defined")

## 3. Run Span Ablation

In [ ]:
# Process all essays
all_results = []
rng = random.Random(RANDOM_SEED)
start_time = time.time()

# Cache tokenization
essay_tokens = {}
for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    essay_tokens[essay['essay_id']] = token_ids

print(f"Tokenized {len(essay_tokens)} essays")
print(f"\nRunning span ablation (5 spans × {len(cohort)} essays = {5*len(cohort)} conditions)...")

for essay in tqdm(cohort, desc="Processing essays"):
    essay_id = essay['essay_id']
    token_ids = essay_tokens[essay_id]
    
    # Run span ablation
    results = run_span_ablation_for_essay(token_ids, essay_id, rng)
    
    # Add essay metadata
    for r in results:
        r['score'] = essay.get('score')
        r['score_bin'] = essay.get('score_bin')
        r['grade'] = essay.get('grade')
        r['token_count'] = len(token_ids)
    
    all_results.extend(results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(cohort):.2f}s/essay)")
print(f"Total result rows: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results shape: {df.shape}")
print(f"\nSpan label counts:")
print(df['span_label'].value_counts())

print(f"\nSample rows:")
df.head(10)

## 4. Group Summaries

In [ ]:
# Summary by score_bin × span_label
GROUP_ORDER = ['low', 'mid', 'high']
SPAN_ORDER = ['early', 'early_mid', 'middle', 'late', 'random']

print("="*80)
print("DELTA NLL BY SCORE_BIN × SPAN_LABEL")
print("="*80)
print("\nHigher delta_nll = removing that span hurts predictions more = span was more important")

summary_rows = []

print(f"\n{'Score Bin':<12} {'Span':<12} {'N':>6} {'Mean ΔNLL':>12} {'SEM':>10} {'Flip Rate':>12}")
print("-"*70)

for score_bin in GROUP_ORDER:
    for span in SPAN_ORDER:
        subset = df[(df['score_bin'] == score_bin) & (df['span_label'] == span)]
        if len(subset) == 0:
            continue
        
        mean_dnll = subset['delta_nll_mean'].mean()
        sem_dnll = subset['delta_nll_mean'].sem()
        mean_flip = subset['flip_rate'].mean()
        
        summary_rows.append({
            'score_bin': score_bin,
            'span_label': span,
            'n': len(subset),
            'delta_nll_mean': mean_dnll,
            'delta_nll_sem': sem_dnll,
            'delta_nll_ci95': 1.96 * sem_dnll,
            'flip_rate_mean': mean_flip,
        })
        
        print(f"{score_bin:<12} {span:<12} {len(subset):>6} {mean_dnll:>12.4f} {sem_dnll:>10.4f} {mean_flip:>12.3f}")

df_summary = pd.DataFrame(summary_rows)

In [ ]:
# Pivot table view
print("\n" + "="*80)
print("PIVOT: Mean ΔNLL by Score Bin (rows) × Span Label (columns)")
print("="*80)

pivot = df.pivot_table(
    values='delta_nll_mean',
    index='score_bin',
    columns='span_label',
    aggfunc='mean'
)[SPAN_ORDER]

print(pivot.round(4))

## 5. Statistical Tests

In [ ]:
# Prepare for regression
df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)
df['span_label'] = pd.Categorical(df['span_label'], categories=SPAN_ORDER, ordered=True)
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()

print("="*80)
print("REGRESSION: delta_nll_mean ~ C(score_bin) + token_count_z + C(span_label) + C(score_bin):C(span_label)")
print("="*80)
print("\nTests whether score_bin effects on delta_nll differ by span location (interaction)")

# Full model with interaction
formula = 'delta_nll_mean ~ C(score_bin) + token_count_z + C(span_label) + C(score_bin):C(span_label)'
model_full = smf.ols(formula, data=df).fit()

print(f"\nFormula: {formula}")
print(f"R²: {model_full.rsquared:.4f}, n={int(model_full.nobs)}")
print(model_full.summary().tables[1])

In [ ]:
# Simpler model without interaction for comparison
print("\n" + "="*80)
print("SIMPLER MODEL: delta_nll_mean ~ C(score_bin) + token_count_z + C(span_label)")
print("="*80)

formula_simple = 'delta_nll_mean ~ C(score_bin) + token_count_z + C(span_label)'
model_simple = smf.ols(formula_simple, data=df).fit()

print(f"\nR²: {model_simple.rsquared:.4f}")
print(model_simple.summary().tables[1])

# Key findings
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

print("\nScore bin effects (vs low):")
for param in ['C(score_bin)[T.mid]', 'C(score_bin)[T.high]']:
    if param in model_simple.params:
        coef = model_simple.params[param]
        pval = model_simple.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

print("\nSpan effects (vs early):")
for span in SPAN_ORDER[1:]:
    param = f'C(span_label)[T.{span}]'
    if param in model_simple.params:
        coef = model_simple.params[param]
        pval = model_simple.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {span}: β={coef:+.4f}, p={pval:.4f} {sig}")

## 6. Visualization

In [ ]:
# Main plot: Span importance by score bin
COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

fig, ax = plt.subplots(figsize=(12, 7))

x_positions = np.arange(len(SPAN_ORDER))
width = 0.25

for i, score_bin in enumerate(GROUP_ORDER):
    means = []
    cis = []
    for span in SPAN_ORDER:
        row = df_summary[(df_summary['score_bin'] == score_bin) & (df_summary['span_label'] == span)]
        if len(row) > 0:
            means.append(row['delta_nll_mean'].values[0])
            cis.append(row['delta_nll_ci95'].values[0])
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    offset = (i - 1) * width
    bars = ax.bar(x_positions + offset, means, width, yerr=cis, 
                  label=score_bin, color=COLORS[score_bin], capsize=3, alpha=0.8)

ax.set_xlabel('Ablated Span Location', fontsize=12)
ax.set_ylabel('Mean ΔNLL (ablated - baseline)', fontsize=12)
ax.set_title('Span Importance by Score Bin\n(Higher = ablating that span hurts predictions more)', fontsize=14, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(['Early\n(0-10%)', 'Early-Mid\n(10-20%)', 'Middle\n(40-50%)', 'Late\n(70-80%)', 'Random\n(10%)'])
ax.legend(title='Score Bin', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'span_importance_by_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Line plot version
fig, ax = plt.subplots(figsize=(10, 6))

for score_bin in GROUP_ORDER:
    means = []
    cis = []
    for span in SPAN_ORDER:
        row = df_summary[(df_summary['score_bin'] == score_bin) & (df_summary['span_label'] == span)]
        if len(row) > 0:
            means.append(row['delta_nll_mean'].values[0])
            cis.append(row['delta_nll_ci95'].values[0])
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    ax.errorbar(x_positions, means, yerr=cis, marker='o', capsize=4,
                label=score_bin, color=COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xlabel('Ablated Span Location', fontsize=12)
ax.set_ylabel('Mean ΔNLL', fontsize=12)
ax.set_title('Context Span Importance Curves by Score Bin', fontsize=14, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(['Early', 'Early-Mid', 'Middle', 'Late', 'Random'])
ax.legend(title='Score Bin', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'span_importance_lines.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Flip rate plot
fig, ax = plt.subplots(figsize=(10, 6))

for score_bin in GROUP_ORDER:
    means = []
    for span in SPAN_ORDER:
        row = df_summary[(df_summary['score_bin'] == score_bin) & (df_summary['span_label'] == span)]
        if len(row) > 0:
            means.append(row['flip_rate_mean'].values[0])
        else:
            means.append(np.nan)
    
    ax.plot(x_positions, means, marker='s', label=score_bin, color=COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xlabel('Ablated Span Location', fontsize=12)
ax.set_ylabel('Flip Rate (top-1 prediction changes)', fontsize=12)
ax.set_title('Top-1 Prediction Flip Rate by Span Location', fontsize=14, fontweight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(['Early', 'Early-Mid', 'Middle', 'Late', 'Random'])
ax.legend(title='Score Bin', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Path(OUTPUT_BASE) / EXPERIMENT / 'flip_rate_by_span.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Results

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)

# Save essay-level long-format results
df.to_csv(output_dir / 'span_ablation_results.csv', index=False)

# Save group summary
df_summary.to_csv(output_dir / 'span_ablation_summary.csv', index=False)

# Save regression results
with open(output_dir / 'regression_span_ablation.txt', 'w') as f:
    f.write("SPAN ABLATION REGRESSION ANALYSIS\n")
    f.write("="*80 + "\n\n")
    f.write("Model with interaction:\n")
    f.write(f"Formula: {formula}\n")
    f.write(model_full.summary().as_text())
    f.write("\n\n" + "="*80 + "\n\n")
    f.write("Simpler model (no interaction):\n")
    f.write(f"Formula: {formula_simple}\n")
    f.write(model_simple.summary().as_text())

# Save pivot table
pivot.to_csv(output_dir / 'span_ablation_pivot.csv')

print(f"\nSaved to {output_dir}/")
print(f"  - span_ablation_results.csv ({len(df)} rows)")
print(f"  - span_ablation_summary.csv")
print(f"  - regression_span_ablation.txt")
print(f"  - span_ablation_pivot.csv")
print(f"  - span_importance_by_score.png")
print(f"  - span_importance_lines.png")
print(f"  - flip_rate_by_span.png")

In [ ]:
# Final interpretation
print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)

print("""
This analysis complements the truncation/half-life approach:

TRUNCATION (existing): How far back does useful context extend?
  - Measures total benefit of having more context
  - Half-life = distance where 50% of benefit is achieved

SPAN ABLATION (new): Which part of prior context drives word choices?
  - Measures importance of specific context regions
  - Higher ΔNLL = that region was more informative

Key questions this answers:
1. Do higher-quality essays depend more on early context (thesis/intro)?
2. Do lower-quality essays depend more on recent/late context (local coherence only)?
3. Is there a score × span interaction (different usage patterns)?

If span importance patterns differ by score_bin, this suggests genuine
differences in long-range structure, not just fluency/difficulty.
""")